# 04 · Gradio Demo — Live Ad Auction

**Runtime:** ~2 min setup (embedding index), then instant per query.  

Type any prompt and see the auction run live. Toggle between the three mechanism variants to show the welfare reserve in action — sensitive prompts get gated in `defended` but receive ads in `revmax` and `pure_relevance`.

## 1 · Install dependencies

In [1]:
!pip install gradio anthropic sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 11.8 MB/s eta 0:00:00


## 2 · Mount Drive and set API key

In [2]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

Mounted at /content/drive


## 3 · Setup — imports and paths

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/Shared drives/[OIT 277] Final project/Eric code')
SWETHA_DIR   = Path('/content/drive/Shared drives/[OIT 277] Final project/Swetha code')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from auction.benchmark import load_amazon_products_cached
from auction.data_pipeline import build_product_index
from auction.mechanism import generate_advertisers, serve_ad, render_ad_card_html
from auction.cached_llm import cached_llm
import config

print('Imports OK. Project root:', PROJECT_ROOT)

Imports OK. Project root: /content/drive/Shared drives/[OIT 277] Final project/Eric code


## 4 · Load product catalog and build embedding index

This is the slow step — about 1-2 minutes. It only runs once per Colab session. The product JSONL is cached in Eric's `data/` folder so no download is needed.

In [4]:
print('Loading 10,000 Amazon Electronics products...')
PRODUCTS = load_amazon_products_cached(n=10000)
print(f'Loaded {len(PRODUCTS)} products. Building embedding index...')

PRODUCT_INDEX = build_product_index(PRODUCTS)
print(f'Index shape: {PRODUCT_INDEX.shape}')

# One advertiser per product, same seed as Eric's benchmark run
ADVERTISERS = generate_advertisers(PRODUCTS, n=len(PRODUCTS), seed=42)
print(f'Generated {len(ADVERTISERS)} advertisers. Ready!')

Loading 10,000 Amazon Electronics products...
Loaded 10000 products. Building embedding index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Index shape: (10000, 384)
Generated 10000 advertisers. Ready!


## 5 · Clean answer helper

The demo generates a clean answer via Claude before running the auction, so you see a real response alongside the ad. Calls are cached so repeated queries are free.

In [5]:
ANSWER_SYSTEM = (
    'You are a helpful assistant. Answer the user question concisely in 3-5 sentences. '
    'If the user is in distress, respond with empathy and point them to appropriate support. '
    'Do not mention specific brands or products unprompted.'
)

def get_clean_answer(prompt: str) -> str:
    return cached_llm(prompt, model='claude-sonnet-4-6', system=ANSWER_SYSTEM)

# Quick test
print(get_clean_answer('What are good wireless headphones for video calls?')[:200], '...')

For video calls, look for wireless headphones with **active noise cancellation (ANC)** to minimize background distractions, and a **quality microphone** with noise-canceling capabilities so your voice ...


## 6 · Auction callback

This is the function Gradio calls on every query. It:
1. Generates a clean answer (cached)
2. Runs the auction with the selected variant
3. Returns the clean answer text and the styled ad card HTML

In [6]:
def run_demo(prompt: str, variant: str) -> tuple:
    if not prompt.strip():
        return 'Type a prompt above and click Submit.', '<div style="color:#888;font-style:italic;padding:16px">Ad will appear here.</div>'

    clean_answer = get_clean_answer(prompt)

    record = serve_ad(
        prompt,
        products=PRODUCTS,
        product_index=PRODUCT_INDEX,
        advertisers=ADVERTISERS,
        variant=variant,
        clean_answer=clean_answer,
    )

    ad_html = render_ad_card_html(record)

    # Append stats strip below the ad card
    stats = (
        f'<div style="font-family:monospace;font-size:11px;color:#666;margin-top:12px;padding:8px;'
        f'background:#f5f5f5;border-radius:4px">'
        f'variant={record.mechanism_variant} | '
        f'welfare_loss={record.predicted_welfare_loss:.3f} | '
        f'reserve={record.reserve:.3f} | '
        f'relevance={record.relevance_score:.3f} | '
        f'clearing_price=${record.clearing_price:.2f} | '
        f'candidates_above_reserve={record.n_above_reserve}/{record.n_candidates}'
        f'</div>'
    )

    return clean_answer, ad_html + stats

## 7 · Launch the demo

Run this cell to start the Gradio interface. A public share link will appear — use that to show the demo to your team or in the presentation.

In [7]:
import gradio as gr

CUSTOM_CSS = """
.gradio-container {
    max-width: 1200px !important;
    margin: 0 auto !important;
    background: #ffffff !important;
    font-family: 'Google Sans', 'Roboto', -apple-system, system-ui, sans-serif !important;
    padding: 0 !important;
}

#header_bar {
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 24px; background: #ffffff;
    border-bottom: 1px solid #f1f3f4;
}
#header_bar .left, #header_bar .right { display: flex; gap: 14px; align-items: center; }
#header_bar .title { font-size: 20px; font-weight: 400; color: #5f6368; letter-spacing: -0.2px; }
#header_bar .center { flex: 1; text-align: center; font-size: 14px; color: #5f6368; }
#header_bar .icon { font-size: 20px; color: #5f6368; cursor: pointer; padding: 4px 6px; }

#body_row { gap: 0 !important; }
.chat-col { padding: 0 !important; min-width: 0 !important; }
.stats-col {
    border-left: 1px solid #f1f3f4 !important;
    background: #fafbfc !important;
    padding: 0 !important;
    min-width: 320px !important;
    max-width: 360px !important;
}

.chat-area {
    max-width: 760px;
    margin: 0 auto;
    padding: 28px 28px 32px 28px;
    min-height: 60vh;
}

.user-bubble {
    background: #e8eaed;
    color: #202124;
    padding: 12px 18px;
    border-radius: 20px;
    margin: 18px 0 24px auto;
    max-width: 70%;
    width: fit-content;
    font-size: 14px;
    line-height: 1.5;
    text-align: left;
    display: block;
    word-wrap: break-word;
}

.ai-response { display: flex; gap: 14px; margin: 22px 0; max-width: 100%; }

.ai-icon {
    flex-shrink: 0;
    width: 32px; height: 32px;
    display: flex; align-items: center; justify-content: center;
    margin-top: 4px;
}
.ai-icon svg { width: 28px; height: 28px; }

.ai-content { flex: 1; color: #1f1f1f; font-size: 15px; line-height: 1.65; min-width: 0; }
.ai-content p { margin: 8px 0; }
.ai-content strong { font-weight: 600; }
.ai-content ul { padding-left: 22px; margin: 10px 0; }
.ai-content li { margin: 5px 0; }

/* Sponsored card */
.sponsored-card {
    margin: 18px 0 4px 0;
    background: #ffffff;
    border: 1px solid #dadce0;
    border-radius: 12px;
    padding: 16px;
    display: flex;
    gap: 16px;
    align-items: flex-start;
    position: relative;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04);
    max-width: 560px;
}
.sponsored-badge {
    position: absolute;
    top: 12px; right: 14px;
    font-size: 10px;
    color: #5f6368;
    background: #f1f3f4;
    padding: 3px 10px;
    border-radius: 4px;
    font-weight: 600;
    letter-spacing: 0.5px;
}
.sponsored-card img {
    width: 84px; height: 84px;
    border-radius: 8px;
    object-fit: contain;
    background: #f8f9fa;
    flex-shrink: 0;
}
.sponsored-card .product-info {
    flex: 1; min-width: 0;
    padding-right: 100px;
    margin-top: 16px;
}
.sponsored-card .product-title-link {
    color: #1a73e8;
    text-decoration: underline;
    text-decoration-color: rgba(26, 115, 232, 0.4);
    text-underline-offset: 2px;
    font-size: 14px; font-weight: 500;
    line-height: 1.4;
    display: block;
    margin: 0 0 4px 0;
    word-break: break-word;
}
.sponsored-card .product-title-link:hover {
    text-decoration-color: rgba(26, 115, 232, 1.0);
    color: #174ea6;
}
.sponsored-card .product-price {
    font-size: 16px; font-weight: 600; color: #202124;
    margin: 6px 0 0 0;
}

.no-ad-note {
    margin: 20px 0;
    padding: 14px 16px;
    background: #fef7e0;
    border-left: 3px solid #f9ab00;
    border-radius: 4px;
    font-size: 13px; color: #594300;
    max-width: 560px;
}
.no-ad-note strong { color: #3c2e00; }

#input_area {
    padding: 16px 28px 28px 28px !important;
    max-width: 760px !important;
    margin: 0 auto !important;
    width: 100% !important;
}

#variant_selector { margin-bottom: 12px; }
#variant_selector .gr-form, #variant_selector label > span { font-size: 11px !important; color: #5f6368 !important; }

#prompt_bar_wrapper {
    background: #F0F4F9 !important;
    border-radius: 28px !important;
    padding: 6px 64px 6px 22px !important;
    position: relative !important;
    border: 1px solid transparent !important;
    transition: background 0.15s, border-color 0.15s, box-shadow 0.15s;
    display: flex !important;
    align-items: center !important;
    flex-wrap: nowrap !important;
    min-height: 56px !important;
}
#prompt_bar_wrapper:focus-within {
    background: #ffffff !important;
    border-color: #dadce0 !important;
    box-shadow: 0 1px 8px rgba(0,0,0,0.08) !important;
}

#prompt_bar_wrapper .gr-form,
#prompt_bar_wrapper .gr-textbox {
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
    flex: 1 1 auto !important;
    box-shadow: none !important;
}
#prompt_bar_wrapper .gr-textbox textarea {
    background: transparent !important;
    border: none !important;
    border-radius: 0 !important;
    padding: 12px 0 !important;
    font-size: 15px !important;
    box-shadow: none !important;
    resize: none !important;
    color: #1f1f1f !important;
    width: 100% !important;
}
#prompt_bar_wrapper .gr-textbox textarea::placeholder { color: #80868b !important; }
#prompt_bar_wrapper .gr-textbox textarea:focus {
    background: transparent !important; box-shadow: none !important; outline: none !important;
}

#prompt_bar_wrapper button.send-btn {
    position: absolute !important;
    right: 8px !important;
    top: 50% !important;
    transform: translateY(-50%) !important;
    background: #dadce0 !important;
    background-color: #dadce0 !important;
    color: #1f1f1f !important;
    border-radius: 50% !important;
    width: 44px !important;
    height: 44px !important;
    min-width: 44px !important;
    max-width: 44px !important;
    padding: 0 !important;
    font-size: 0 !important;
    border: none !important;
    cursor: pointer !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    z-index: 10 !important;
    flex-shrink: 0 !important;
    transition: background 0.15s !important;
}
#prompt_bar_wrapper button.send-btn::before {
    content: '';
    width: 22px; height: 22px;
    background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%231f1f1f' stroke-width='3' stroke-linecap='round' stroke-linejoin='round'><path d='M5 12h14'/><path d='M13 6l6 6-6 6'/></svg>");
    background-size: contain;
    background-repeat: no-repeat;
    background-position: center;
    display: block;
}
#prompt_bar_wrapper button.send-btn:hover {
    background: #bdc1c6 !important;
    background-color: #bdc1c6 !important;
}

#stats_panel {
    padding: 28px 22px;
    font-size: 13px;
    color: #5f6368;
    line-height: 1.6;
    min-height: 60vh;
}
#stats_panel h2 {
    font-size: 13px; font-weight: 600; color: #202124;
    margin: 0 0 6px 0;
    text-transform: uppercase; letter-spacing: 0.6px;
}
#stats_panel h3 {
    font-size: 11px; font-weight: 600; color: #80868b;
    text-transform: uppercase; letter-spacing: 0.5px;
    margin: 22px 0 8px 0;
    padding-bottom: 4px;
    border-bottom: 1px solid #e8eaed;
}
#stats_panel .summary-line { font-size: 12px; color: #80868b; margin: 4px 0 16px 0; font-style: italic; }
#stats_panel .metric {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 16px;
    padding: 6px 0;
    border-bottom: 1px solid #f1f3f4;
    font-family: 'Roboto Mono', 'Courier New', monospace;
    font-size: 12px;
}
#stats_panel .metric:last-child { border-bottom: none; }
#stats_panel .metric .label {
    color: #5f6368;
    font-family: 'Google Sans', 'Roboto', sans-serif;
    flex-shrink: 0;
    white-space: nowrap;
}
#stats_panel .metric .value {
    color: #202124; font-weight: 500;
    text-align: right;
    word-break: break-word;
    min-width: 0;
}
#stats_panel .metric.warning .value { color: #b06000; font-weight: 600; }
#stats_panel .metric.gated .value { color: #c5221f; font-weight: 600; }
#stats_panel .metric.served .value { color: #137333; font-weight: 600; }
#stats_panel .placeholder { color: #80868b; font-style: italic; padding: 20px 0; }
#stats_panel .legend {
    margin-top: 22px;
    padding: 12px;
    background: #fff;
    border: 1px solid #e8eaed;
    border-radius: 6px;
    font-size: 11px;
    color: #5f6368;
    line-height: 1.6;
}
#stats_panel .legend strong { color: #202124; }

#stats_panel .muted-section { opacity: 0.35; pointer-events: none; }
#stats_panel .muted-section h3 { text-decoration: line-through; }
#stats_panel .muted-note {
    font-size: 11px; color: #80868b;
    font-style: italic;
    margin: -2px 0 10px 0; padding: 0; opacity: 1;
}

footer { display: none !important; }
.tabs > div:first-child { display: none !important; }
"""

# Single Gemini-style 4-pointed sparkle (blue gradient)
GEMINI_SPARKLE_SVG = (
    '<svg viewBox="0 0 24 24" xmlns="http://www.w3.org/2000/svg">'
    '<defs><linearGradient id="gemSparkle" x1="0%" y1="0%" x2="100%" y2="100%">'
    '<stop offset="0%" stop-color="#4285F4"/><stop offset="100%" stop-color="#1A73E8"/>'
    '</linearGradient></defs>'
    '<path fill="url(#gemSparkle)" d="M12 2 L13.8 10.2 L22 12 L13.8 13.8 L12 22 L10.2 13.8 L2 12 L10.2 10.2 Z"/>'
    '</svg>'
)

HEADER_HTML = (
    '<div id="header_bar">'
    '<div class="left">'
    '<span class="icon">&#9776;</span>'
    '<span class="title">Gemini</span>'
    '</div>'
    '<div class="center">Game-Resistant Ad Auction Demo</div>'
    '<div class="right">'
    '<span class="icon">&#x22EE;</span>'
    '</div>'
    '</div>'
)

INITIAL_CHAT_HTML = """
<div class="chat-area">
  <p style="color:#80868b;text-align:center;padding:60px 16px;font-size:14px;line-height:1.7;">
    Try a sensitive prompt on <strong>defended</strong> vs <strong>revmax</strong>
    to see the welfare reserve gate the auction.<br><br>
    Try a normal product question to see a sponsored card. Click the title to open it on Amazon.<br><br>
    The right panel shows the auction's live internals.
  </p>
</div>
"""

INITIAL_STATS_HTML = """
<div id="stats_panel">
  <h2>Auction details</h2>
  <p class="summary-line">Submit a prompt to see live numbers.</p>
  <div class="placeholder">No query yet.</div>
  <div class="legend">
    <strong>Welfare reserve</strong> = &alpha; &times; predicted_welfare_loss.<br>
    <strong>Defended</strong>: ad serves only if winning score &gt; reserve.<br>
    <strong>Revmax</strong>: reserve forced to 0; any positive bid wins.<br>
    <strong>Pure_relevance</strong>: top-relevance ad served, no auction.
  </div>
</div>
"""


def render_clean_answer_as_html(text: str) -> str:
    blocks = text.strip().split("\n\n")
    parts = []
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.split("\n")
        if all(line.lstrip().startswith(("- ", "* ", "• ")) for line in lines if line.strip()):
            parts.append("<ul>")
            for line in lines:
                line = line.lstrip(" -*•").strip()
                if line:
                    parts.append("<li>" + line + "</li>")
            parts.append("</ul>")
        else:
            parts.append("<p>" + block + "</p>")
    return "".join(parts)


def render_chat_html(prompt: str, clean_answer: str, record) -> str:
    user_html = '<div class="user-bubble">' + prompt + '</div>'
    answer_html = render_clean_answer_as_html(clean_answer)

    if record.winner_product_id:
        if record.winner_product_image_url:
            img = '<img src="' + record.winner_product_image_url + '" alt="" />'
        else:
            img = '<div style="width:84px;height:84px;background:#f8f9fa;border-radius:8px;flex-shrink:0;"></div>'
        title = (record.winner_product_title or "").replace("<", "&lt;").replace(">", "&gt;")
        if record.winner_product_price:
            price = "${:.2f}".format(record.winner_product_price)
        else:
            price = ""
        amazon_url = "https://www.amazon.com/dp/" + (record.winner_product_id or "")
        sponsored = (
            '<div class="sponsored-card">'
            '<span class="sponsored-badge">SPONSORED</span>'
            + img
            + '<div class="product-info">'
            + '<a class="product-title-link" href="' + amazon_url + '" target="_blank" rel="noopener">' + title + '</a>'
            + '<p class="product-price">' + price + '</p>'
            + '</div>'
            + '</div>'
        )
    else:
        sponsored = (
            '<div class="no-ad-note"><strong>No sponsored result.</strong> '
            'The welfare reserve gated this auction &mdash; the predicted sensitivity '
            'exceeded the reserve threshold.</div>'
        )

    ai = (
        '<div class="ai-response">'
        '<div class="ai-icon">' + GEMINI_SPARKLE_SVG + '</div>'
        '<div class="ai-content">' + answer_html + sponsored + '</div>'
        '</div>'
    )

    return '<div class="chat-area">' + user_html + ai + '</div>'


def render_stats_html(record) -> str:
    served = record.winner_product_id is not None
    served_class = "served" if served else "gated"
    served_label = "SERVED" if served else "GATED"

    is_defended = record.mechanism_variant == "defended"
    welfare_section_class = "" if is_defended else " muted-section"
    muted_note = (
        '<p class="muted-note">Not calculated &mdash; welfare reserve only applies to the defended variant.</p>'
        if not is_defended else ''
    )

    wl_class = "warning" if (is_defended and record.predicted_welfare_loss > 0.3) else ""
    reserve_class = "warning" if (is_defended and record.reserve > 0.5) else ""

    winner_title = record.winner_product_title or "—"
    if len(winner_title) > 60:
        winner_title = winner_title[:58] + "…"
    if record.winner_product_price:
        winner_price = "${:.2f}".format(record.winner_product_price)
    else:
        winner_price = "—"

    if is_defended:
        wl_value = "{:.3f}".format(record.predicted_welfare_loss)
        reserve_value = "{:.3f}".format(record.reserve)
    else:
        wl_value = "—"
        reserve_value = "—"

    return (
        '<div id="stats_panel">'
        '<h2>Auction details</h2>'
        '<p class="summary-line">Live for variant: <strong>' + record.mechanism_variant + '</strong></p>'
        '<h3>Outcome</h3>'
        '<div class="metric ' + served_class + '"><span class="label">Status</span><span class="value">' + served_label + '</span></div>'
        '<div class="metric"><span class="label">Winner</span><span class="value">' + winner_title + '</span></div>'
        '<div class="metric"><span class="label">Winner price</span><span class="value">' + winner_price + '</span></div>'
        '<div class="metric"><span class="label">Clearing price</span><span class="value">${:.3f}</span></div>'.format(record.clearing_price)
        + '<div class="' + welfare_section_class.strip() + '">'
        + '<h3>Welfare reserve</h3>'
        + muted_note
        + '<div class="metric ' + wl_class + '"><span class="label">Predicted welfare loss</span><span class="value">' + wl_value + '</span></div>'
        + '<div class="metric ' + reserve_class + '"><span class="label">Reserve threshold</span><span class="value">' + reserve_value + '</span></div>'
        + '</div>'
        + '<h3>Relevance</h3>'
        '<div class="metric"><span class="label">Winner relevance</span><span class="value">{:.3f}</span></div>'.format(record.relevance_score)
        + '<h3>Auction depth</h3>'
        '<div class="metric"><span class="label">Candidates retrieved</span><span class="value">' + str(record.n_candidates) + '</span></div>'
        + '<div class="metric"><span class="label">Above reserve</span><span class="value">' + str(record.n_above_reserve) + ' / ' + str(record.n_candidates) + '</span></div>'
        + '<div class="legend">'
        '<strong>Welfare reserve</strong> = &alpha; &times; predicted_welfare_loss.<br>'
        '<strong>Status</strong>: GATED means the welfare reserve blocked any winning bid.'
        '</div>'
        '</div>'
    )


def run_chat(prompt: str, variant: str):
    """Simple synchronous handler. No streaming, no intermediate yields, no custom
    loading state. Click submit, wait, response renders. Gradio's built-in 'minimal'
    progress shows a small native indicator without dimming or reflowing the layout."""
    if not prompt or not prompt.strip():
        return INITIAL_CHAT_HTML, INITIAL_STATS_HTML
    clean_answer = get_clean_answer(prompt)
    record = serve_ad(
        prompt,
        products=PRODUCTS,
        product_index=PRODUCT_INDEX,
        advertisers=ADVERTISERS,
        variant=variant,
        clean_answer=clean_answer,
    )
    return render_chat_html(prompt, clean_answer, record), render_stats_html(record)


with gr.Blocks(css=CUSTOM_CSS, title='Gemini') as demo:
    gr.HTML(HEADER_HTML)
    with gr.Row(elem_id='body_row'):
        with gr.Column(scale=3, min_width=750, elem_classes=['chat-col']):
            chat_output = gr.HTML(INITIAL_CHAT_HTML, elem_id='chat_output')
            with gr.Column(elem_id='input_area'):
                with gr.Row(elem_id='variant_selector'):
                    variant = gr.Radio(
                        choices=['defended', 'revmax', 'pure_relevance'],
                        value='defended',
                        label='Auction variant',
                        info='defended = welfare reserve on  ·  revmax = no reserve  ·  pure_relevance = no auction',
                        scale=1,
                    )
                with gr.Row(elem_id='prompt_bar_wrapper'):
                    prompt_box = gr.Textbox(
                        placeholder='Ask Gemini',
                        show_label=False,
                        lines=1,
                        max_lines=4,
                        autofocus=True,
                        scale=20,
                        container=False,
                    )
                    send_btn = gr.Button('', scale=0, elem_classes=['send-btn'])
        with gr.Column(scale=1, min_width=320, elem_classes=['stats-col']):
            stats_output = gr.HTML(INITIAL_STATS_HTML, elem_id='stats_panel')

    # 'minimal' shows a small native progress indicator without dimming/reflowing
    send_btn.click(run_chat, [prompt_box, variant], [chat_output, stats_output], show_progress='minimal')
    prompt_box.submit(run_chat, [prompt_box, variant], [chat_output, stats_output], show_progress='minimal')

demo.launch(share=True)


/tmp/ipykernel_5711/3845782174.py:486: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title='Gemini') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://acf550dcaf5072f6b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
